# Forge Qwen Training Notebook

This notebook is for the OpenEnv Hackathon submission.

It shows how to run Forge in Colab:

1. install the package
2. choose Qwen model mode
3. run the training loop
4. generate plots
5. package outputs for download or Hugging Face upload

Start with the small 1.5B Qwen model. After that works, try the 7B model with 4-bit loading.

In [ ]:
# If running in Colab, uncomment these install lines.
# !pip install -U pip
# !pip install -e ".[train,dev]"
# !pip install -U torch --index-url https://download.pytorch.org/whl/cu121

import os

# Use stub for a quick CPU check.
# Use transformers for a real Qwen run on Colab T4.
os.environ["FORGE_LLM_BACKEND"] = "stub"
os.environ["FORGE_LLM_MODEL"] = "Qwen/Qwen2.5-1.5B-Instruct"
os.environ["FORGE_TOOL_ISOLATION"] = "subprocess"
os.environ["FORGE_LLM_JUDGE"] = "0"

## Optional: real Qwen model settings

For a real Colab run, change `FORGE_LLM_BACKEND` to `transformers`.

For T4 GPU:

- safer first run: `Qwen/Qwen2.5-1.5B-Instruct`
- larger run: `Qwen/Qwen2.5-7B-Instruct` with `FORGE_LLM_LOAD_IN_4BIT=1`

Unsloth is for a future stronger version where we fine-tune model weights. This notebook currently improves the playbook, not Qwen weights.

In [ ]:
# Real model option for Colab T4. Uncomment to use it.
# os.environ["FORGE_LLM_BACKEND"] = "transformers"
# os.environ["FORGE_LLM_MODEL"] = "Qwen/Qwen2.5-1.5B-Instruct"
# os.environ["FORGE_LLM_LOAD_IN_4BIT"] = "0"

# Larger model option. Uncomment instead of the 1.5B option if you have memory headroom.
# os.environ["FORGE_LLM_BACKEND"] = "transformers"
# os.environ["FORGE_LLM_MODEL"] = "Qwen/Qwen2.5-7B-Instruct"
# os.environ["FORGE_LLM_LOAD_IN_4BIT"] = "1"

In [ ]:
from training.grpo_config import ForgeTrainConfig
from training.train_forge import run_training

cfg = ForgeTrainConfig(
    model_name=os.environ["FORGE_LLM_MODEL"],
    llm_backend=os.environ["FORGE_LLM_BACKEND"],
    max_steps=3,
)

metrics_path = run_training(cfg, smoke=False)
metrics_path

In [ ]:
from outputs.plots.generate_plots import generate

generate(metrics_path)
print("Plots written under outputs/plots")

In [ ]:
from pathlib import Path
from IPython.display import Image, display

for path in [
    "outputs/plots/01_val_accuracy.png",
    "outputs/plots/03_reward_components_stacked.png",
    "outputs/plots/07_baseline_vs_trained.png",
    "outputs/plots/08_training_loss_proxy.png",
]:
    p = Path(path)
    if p.exists():
        print(path)
        display(Image(filename=str(p)))

In [ ]:
# Download outputs from Colab.
# from google.colab import files
# !zip -r forge_outputs.zip outputs
# files.download("forge_outputs.zip")